In [ ]:
import pandas as pd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.mathtext import _mathtext as mathtext
mathtext.FontConstantsBase.sup1 = 0.45

from scipy.stats import gaussian_kde
from sklearn.metrics import mean_squared_error
import cartopy.crs as ccrs

In [ ]:
# Define a function to handle nodata values
def extract_raster_values(lat, lon, raster):
    coords = [(lon, lat)]
    values = [x[0] for x in raster.sample(coords)]
    if values[0] == raster.nodata:
        return np.nan
    return values[0]

# Load the data
file_path_1 = '../../data/supplementary_figures/model_results/AGB_upscale_2020_GB_1pctdeg.csv'
data_1 = pd.read_csv(file_path_1)
guo_path_1 = '../../data/supplementary_figures/cpr2other/Guo/AGB_Guo_1pctdeg.tif'
guo_raster_1 = rasterio.open(guo_path_1)
data_1['Guo'] = data_1.apply(lambda row: extract_raster_values(row['Lat'], row['Lon'], guo_raster_1), axis=1)

# Load the data
file_path_2 = '../../data/supplementary_figures/model_results/AGB_upscale_2020_GB_halfdeg.csv'
data_2 = pd.read_csv(file_path_2)
guo_path_2 = '../../data/supplementary_figures/cpr2other/Guo/AGB_Guo_halfdeg.tif'
guo_raster_2 = rasterio.open(guo_path_2)
data_2['Guo'] = data_2.apply(lambda row: extract_raster_values(row['Lat'], row['Lon'], guo_raster_2), axis=1)

In [ ]:
temp_1 = data_1.iloc[:, :4].dropna()
X = temp_1['AGB']
Y = temp_1['Guo']

temp_2 = data_2.iloc[:, :4].dropna()
Z = temp_2['Guo'] - temp_2['AGB']

# Set global font to Helvetica
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
# Set the mathtext font to Helvetica
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

fig = plt.figure(figsize=(12, 7), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=[4, 3], width_ratios=[1, 2])

ax1 = fig.add_subplot(grid[0, 0])

# 使用 pcolormesh 绘制正方形网格


xi = np.linspace(X.min(), X.max(), 100)
yi = np.linspace(Y.min(), Y.max(), 100)
zi, _, _ = np.histogram2d(X, Y, bins=(xi, yi))

density_threshold = 1
zi[zi < density_threshold] = np.nan
ax1.pcolormesh(xi, yi, zi.T, cmap='Blues')

cbar_ax1 = fig.add_axes([0.27, 0.55, 0.02, 0.18])  # 控制色带位置和大小
cbar1 = plt.colorbar(ax1.collections[0], cax=cbar_ax1, orientation='vertical')
cbar1.ax.set_title('Density', fontsize=14, pad=10)

ax1.plot(np.linspace(0, 600, 100), np.linspace(0, 600, 100), '--', linewidth=1, color='#0072BD')
b = np.polyfit(X, Y, 1)
ax1.plot(np.linspace(0, 600, 100), b[0] * np.linspace(0, 600, 100) + b[1], 'r-', linewidth=1, alpha=0.5)
R2 = np.corrcoef(X, Y)[0, 1]**2
RMSE = np.sqrt(mean_squared_error(Y, X))
PBIAS = np.sum(Y - X) / np.sum(X) * 100
ax1.text(50, 400, f'y = {b[0]:.2f}x + {b[1]:.2f}\nR$^2$ = {R2:.2f}\nPBIAS = {PBIAS:.2f}%')
ax1.set_xlim([0, 601])
ax1.set_ylim([0, 601])
ax1.set_xlabel('AGB density from this study (Mg DM ha$^{-1}$)')
ax1.set_ylabel('AGB density from\nHu et al. (Mg DM ha$^{-1}$)')
ax1.set_xticks([0, 200, 400, 600])
ax1.set_yticks([0, 200, 400, 600])
ax1.tick_params(axis='both', direction='out')
ax1.set_title('a', fontweight='bold', loc='left')
ax1.set_aspect('equal', adjustable='box')

ax2 = fig.add_subplot(grid[0, 1])
# 第二个子图（核密度估计图）
xi = np.linspace(0, 400, 100)
kde_X = gaussian_kde(X)
f1 = kde_X(xi) * 100
b1 = np.mean(X)
c1 = np.median(X)

yi = np.linspace(0, 400, 100)
kde_Y = gaussian_kde(Y)
f2 = kde_Y(yi) * 100
b2 = np.mean(Y)
c2 = np.median(Y)

ax2.plot(xi, f1, linewidth=1, linestyle='-', color=(105/255, 178/255, 111/255))
ax2.plot(yi, f2, linewidth=1, linestyle='-', color=(255/255, 190/255, 77/255))
ax2.set_xlabel('AGB density (Mg DM ha$^{-1}$)')
ax2.set_ylabel('Density (%)')
ax2.set_xlim([-20, 400])
ax2.set_ylim([-0.1, 1])
ax2.set_xticks([0, 200, 400])
ax2.set_yticks([0, 0.5, 1])
ax2.legend(['This study', 'Hu et al.'], loc='upper right', frameon=False)
ax2.set_title('b', fontweight='bold', loc='left')
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)

ax3 = fig.add_subplot(grid[1, :], projection=ccrs.PlateCarree())
ax3.set_extent([-180, 180, -40, 40], crs=ccrs.PlateCarree())
ax3.set_title('c', loc='left', fontweight='bold')
ax3.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=0)
scatter = ax3.scatter(temp_2['Lon'], temp_2['Lat'], c=Z, cmap='RdBu_r', s=5, edgecolor='none', transform=ccrs.PlateCarree(), zorder=1)
scatter.set_clim(-100, 100)

# 添加色带
cbar_ax3 = fig.add_axes([0.14, 0.08, 0.15, 0.02])  # 控制色带位置和大小
cbar3 = plt.colorbar(scatter, cax=cbar_ax3, orientation='horizontal')

cbar3.ax.set_title('AGB density from Hu et al.\nminus AGB density from\nthis study (Mg DM ha$^{-1}$)', fontsize=14, pad=10)
cbar3.ax.tick_params(labelsize=14)

plt.savefig('../../figures/supplementary/figS04_compare_hu.png', dpi=400)
plt.show()